In [2]:
# Setup the Jupyter version of Dash
from datetime import datetime
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html, no_update
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# import AnimalShelter object from CRUD module
from CRUD_Python_Module import AnimalShelter

FILTER_QUERIES = {
    "WATER_RESCUE": {
        "animal_type": "Dog",
        "breed": {
            "$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever",
                "Newfoundland",
            ]
        },
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
    },
    "MOUNTAIN_RESCUE": {
        "animal_type": "Dog",
        "breed": {
            "$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler",
            ]
        },
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
    },
    "DISASTER_RESCUE": {
        "animal_type": "Dog",
        "breed": {
            "$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler",
            ]
        },
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
    },
}
###########################
# Data Manipulation / Model
###########################

# likely need more variables for your constructor to handle the hostname and port of the MongoDB
# server, and the database and collection names

username = "aacuser"
password = "88ww77ggWOW"
db = AnimalShelter(username, password)


# create dataframe of all animals
df = pd.DataFrame.from_records(db.read({}))
# drop the _id column from the dataframe
df.drop(columns=["_id"], inplace=True)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash("Location Dashboard for Grazioso Salvare")

# load and encode image so it can be displayed in dash
image_filename = "./Grazioso Salvare Logo.png" 
encoded_image = base64.b64encode(open(image_filename, 'rb').read())
logo_src = "data:image/png;base64,{}".format(encoded_image.decode())

# dashboard layout
app.layout = html.Div(
    [
        html.Div(
            [
                html.A(
                    html.Img(src=logo_src, style={'height':'100px'}),
                    href="https://www.snhu.edu/",
                    target="_blank"
                ),
                # page header
                html.Div(
                    children = [
                        html.H1("Grazioso Salvare Dashboard"),
                        html.H1(f"SNHU CS340 Project Two - Grant Gimer {datetime.now().strftime('%m/%d/%Y')}"),
                
                    ],
                    style={
                        "display": "flex",
                        "flexDirection": "row",
                        "justifyContent": "space-between",
                        "alignItems": "center",
                        "width": "100%",
                        "padding": "5px 0",
                    }
                )
            ],
            style={
                "display": "flex",
                "justifyContent": "space-between",
                "alignItems": "center",
                "width": "100%",
                "padding": "5px 0",
            },
        ),
        html.Div(id="hidden-div", style={"display": "none"}),

        # filter buttons
        html.Div(
            [
                html.Label("Filter:", style={"font-weight": "bold"}),
                dcc.RadioItems(
                id="filter-selection",
                options=[ 
                    {"label": "Water Rescue", "value": "WATER_RESCUE"},
                    {"label": "Mountain Rescue or Wilderness Rescue", "value": "MOUNTAIN_RESCUE"},
                    {"label":   "Disaster Rescue or Individual Tracking", "value": "DISASTER_RESCUE"},
                    {"label": "Reset", "value": "ALL_ANIMALS"},
                ],
                value="ALL_ANIMALS",  # Initial selected value
                inline=True,
                labelStyle={
                    "display": "flex",
                    "justifyContent": "left",
                    "margin-right": "24px",
                    "margin-left": "24px",
                }
            )
            ],
        ),
        # main data display table
        dash_table.DataTable(
            id="datatable-id",
            columns=[
                {"name": i, "id": i, "deletable": False, "selectable": True}
                for i in df.columns
            ],
            data=df.to_dict("records"),
            filter_action="native",
            sort_action="native",
            sort_mode="multi",
            page_action="native",
            page_current=0,
            page_size=10,
            column_selectable=False,
            selected_columns=[],
            row_selectable="single",
            row_deletable=False,
            selected_rows=[0],
            style_table={"overflowX": "auto"},
            style_cell={
                "textAlign": "left",
                "padding": "2px",
                "minWidth": "50px",
                "maxWidth": "250px",
            },
            style_header={
                "textAlign": "left",
                "backgroundColor": "#D2D6A9",
                "font-weight": "bold",
            },
        ),
        # graphs and map, laid out side by side
        html.Div(
            children=[
                html.Div(
                    id='graph-id',
                    style={"flex": "1", "minHeight": "0vh", "width": "50%"}
                ),
                html.Div(
                    id="map-id",
                    style={"flex": "1", "minHeight": "0vh", "width": "50%"},
                ),
            ],
            style={
                "display": "flex",
                "flexDirection": "row",
                "height": "50vh",
                "maxHeight": "50vh",
                "overflow": "hidden",
                "boxSizing": "border-box",
                "padding": "10px",
            },
        ),
    ],
)

# callback to apply filters to the data table when a filter button is clicked
@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-selection", "value")]
)
def apply_filters(filter_name):
    if filter_name == "ALL_ANIMALS":
        read_result = db.read({})
    else:
        read_result = db.read(FILTER_QUERIES[filter_name])
    new_df = pd.DataFrame.from_records(read_result)
    if "_id" in new_df.columns:
        new_df.drop(columns=["_id"], inplace=True)
    return new_df.to_dict('records')



# update map when a row is clicked in any cell
@app.callback(
    Output("datatable-id", "selected_rows"),
    Input("datatable-id", "active_cell"),
    # use state and derived viewport indices to get the actual row index only on input so page changes don't change map
    State("datatable-id", "derived_viewport_indices"),
    prevent_initial_call=True,
)
def update_selected_row(active_cell, viewport_indices):
    if active_cell is None or not viewport_indices:
        return no_update
    return [viewport_indices[active_cell["row"]]]


# callback to update graph with selected row data
@app.callback(
    Output  ("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data"),]
)
def update_graph(viewData):
    if not viewData:
        return no_update
    dff = pd.DataFrame.from_dict(viewData)
    return dcc.Graph(
        figure=px.histogram(
            dff,
            x="age_upon_outcome_in_weeks",
            y="breed",
            color="breed",
            labels={
                "age_upon_outcome_in_weeks": "Age Upon Outcome in Weeks",
                "breed": "Breed",
            },
            title="Age Distribution of Animals by Breed"
        ),
    )


# callback to update map with selected row data
@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows"),
    ],
)
def update_map(viewData, index):
    if not viewData:
        return no_update
    dff = pd.DataFrame.from_dict(viewData)

    # check if a row was clicked/selected and fall back to first row if none
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # get the animal information from the selected row
    lat = dff.iloc[row]['location_lat']
    lon = dff.iloc[row]['location_long']
    breed = dff.iloc[row]['breed']
    name = dff.iloc[row]['name']
    if breed == '':
        breed = 'Unknown Breed'
    if name == '':
        name = 'Unnamed Animal'
    animal_type = dff.iloc[row]['animal_type']
    age = f"{round(float(dff.iloc[row]['age_upon_outcome_in_weeks']), 0)} weeks old"
    outcome = dff.iloc[row]['outcome_type']
    outcome_subtype = dff.iloc[row]['outcome_subtype']
    color = dff.iloc[row]['color']
    sex = dff.iloc[row]['sex_upon_outcome']

    # create and return the leaflet map
    return dl.Map(
        style={'width': '100%', 'height': '100%'},
        center=[lat, lon],
        zoom=12,
        children=[
            dl.TileLayer(id='base-layer-id'),
            dl.Marker(
                position=[lat, lon],
                children=[
                    dl.Tooltip(name + ': ' + breed),
                    dl.Popup([
                        #formatted with all data
                        html.H3(name),
                        html.P(f'Age: {age}'),
                        html.P(f'Sex: {sex}'),
                        html.P(f'Animal Type: {animal_type}'),
                        html.P(f'Breed: {breed}'),
                        html.P(f'Color: {color}'),
                        html.P(f'Outcome: {outcome}'),
                        html.P(f'Outcome Subtype: {outcome_subtype}'),
                        html.P(f'Location: [{lat}, {lon}]'),
                        ]),
                ],
            ),
        ],
    )

# Run app and display result in jupyterlab mode
app.run_server()


Dash app running on https://touchsabine-comedymango-3000.codio.io/proxy/8050/
